# UTH Computer Vision Coursework — Topic 11
## Notebook 01: Exploratory Data Analysis (EDA) on Real Faces (FFHQ & CelebA-HQ)
- **Nhóm thực hiện**: Nhóm 4
- **Đề tài**: *Stable Diffusion-Based Synthetic Facial Dataset Generation for Deepfake Detection*
- **Mục tiêu**: Khảo sát phân bố ảnh khuôn mặt thật từ hai nguồn chuẩn hóa FFHQ và CelebA-HQ, bao gồm độ tuổi, giới tính và phụ kiện che mặt (Glasses/Occlusion).


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Import modular classes and config from src
from src.common.config import load_config
from src.data.image_schema import ImageMetadata
from src.data.splits import normalize_demographics


### 1. Nạp và kiểm tra dữ liệu thô chuẩn hóa (`paths.raw_manifest`)


In [ ]:
cfg = load_config(Path('configs/dataset_factory.yaml'))
manifest_path = Path(cfg['paths']['raw_manifest'])
records = []
with open(manifest_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            records.append(ImageMetadata.from_dict(json.loads(line)))

print(f'Tổng số ảnh khuôn mặt thật đã nạp: {len(records)}')
if records:
    print('Ví dụ bản ghi đầu tiên:', json.dumps(records[0].to_dict(), indent=2))


### 2. Thống kê phân bố nhân khẩu học (Tuổi, Giới tính, Kính)


In [ ]:
datasets = {}
age_groups = {}
genders = {}
glasses = {}

for r in records:
    d = r.demographics or {}
    src = d.get('dataset', 'unknown')
    datasets[src] = datasets.get(src, 0) + 1
    
    norm_d = normalize_demographics(d)
    age = norm_d['age_group']
    age_groups[age] = age_groups.get(age, 0) + 1
    
    g = norm_d['gender']
    genders[g] = genders.get(g, 0) + 1
    
    gl = norm_d['glasses']
    glasses[gl] = glasses.get(gl, 0) + 1

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].bar(list(datasets.keys()), list(datasets.values()), color='#3498db')
axes[0].set_title('Nguồn Dữ Liệu Gốc')
axes[0].set_ylabel('Số lượng ảnh')

axes[1].bar(list(genders.keys()), list(genders.values()), color='#e74c3c')
axes[1].set_title('Phân Bố Giới Tính')

axes[2].bar(list(glasses.keys()), list(glasses.values()), color='#2ecc71')
axes[2].set_title('Phụ Kiện Che Mặt (Glasses/Occlusion)')
axes[2].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()


### 3. Trực quan hóa mẫu ảnh khuôn mặt thật từ FFHQ và CelebA-HQ


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
ffhq_samples = [r for r in records if (r.demographics or {}).get('dataset') == 'ffhq'][:4]
celeba_samples = [r for r in records if (r.demographics or {}).get('dataset') == 'celeba_hq'][:4]

for i, r in enumerate(ffhq_samples):
    ax = axes[0, i]
    if r.file_path and Path(r.file_path).exists():
        im = Image.open(r.file_path).convert('RGB')
        ax.imshow(im)
    demo = r.demographics or {}
    ax.set_title(f'FFHQ: {demo.get("age_group")}, {demo.get("gender")}', fontsize=9)
    ax.axis('off')

for i, r in enumerate(celeba_samples):
    ax = axes[1, i]
    if r.file_path and Path(r.file_path).exists():
        im = Image.open(r.file_path).convert('RGB')
        ax.imshow(im)
    demo = r.demographics or {}
    ax.set_title(f'CelebA: {demo.get("gender")}', fontsize=9)
    ax.axis('off')

plt.suptitle('Mẫu Khuôn Mặt Thật: Hàng 1 (FFHQ) vs Hàng 2 (CelebA-HQ)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
